# Otimização de Equipes em Pokémon Draft Leagues
**Recomendação sequencial e montagem de times via Programação Linear Inteira (MILP)**

Notebook companheiro do trabalho de Otimização (Eduardo Freire e João Pedro Cotrim). Ele percorre o projeto de ponta a ponta, **reutilizando o código que você já tem** (`model1.py`, `model2.py`, `sensibility.py`) em vez de duplicá-lo:

1. **Dados**: custos e estatísticas (Smogon) e uma exploração rápida do pool.
2. **Tipos**: tabela de eficácia, `Advantage` e `Resist` com exemplo passo a passo.
3. **Modelo**: formulação MILP do draft e do duelo.
4. **Coeficientes** $c_i = \alpha v_i + \beta d_i + \gamma r_i$ para um conjunto de adversários.
5. **Modo Game**: melhor time completo contra times adversários prontos (com Top-$k$).
6. **Modo Draft**: simulação de um draft inteiro, rodada a rodada.
7. **Sensibilidade** aos pesos $(\alpha,\beta,\gamma)$ com várias sementes.
8. **Duelo** (`model2.py`): escolher $K$ do roster contra um adversário, e a observação da separabilidade.
9. **Discussão**: limitações e extensões.

**Pré-requisitos.** Na mesma pasta do notebook: `model1.py`, `model2.py` e `sensibility.py`.
Os dados (`costs_pokemon.csv` e `stats_pokemon.csv`) já estão **embutidos** no notebook, na célula recolhida logo após a configuração; não é preciso ter os CSVs em disco.
Bibliotecas: `pip install pandas numpy scipy matplotlib`.

## 1. Configuração
Todos os parâmetros do experimento ficam nesta célula. Altere à vontade e rode o notebook de novo (`Kernel > Restart & Run All`).

In [ ]:
import base64
import gzip
import io
import sys
import itertools
from collections import Counter
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

sys.path.insert(0, str(Path.cwd()))   # model1.py, model2.py e sensibility.py na mesma pasta

from model1 import (TYPES, Config, advantage, best_teams, find_pokemon, load_data,
                    objective_coeffs, opponent_pick, random_team, recommend,
                    resistance, solve, type_multiplier)
from model2 import best_team
from sensibility import METRIC_COLS, make_scenarios, team_metrics

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COLORS = ["#2b6cb0", "#dd6b20", "#2f855a", "#805ad5"]

In [ ]:
# ---- dados ----
USE_EMBEDDED_DATA = True            # True: usa os CSVs embutidos neste notebook
COSTS_CSV = "costs_pokemon.csv"     # só usados se USE_EMBEDDED_DATA = False
STATS_CSV = "stats_pokemon.csv"

# ---- regras da liga ----
N_PLAYERS   = 4        # n = você + (n-1) adversários
G           = 6        # Pokémon por time
C           = 80       # orçamento total de pontos
E           = 1        # máximo de Mega evoluções por time
ALLOW_UBERS = False    # libera Uber/AG?
BAN_ILLEGAL = False    # remove o tier Illegal do pool? (cuidado: todas as Megas estão no tier Illegal)
ATTACK_MODE = "max"    # v_i = max(Atk, SpA) ("max"), só Atk ("atk") ou só SpA ("spa")

# ---- pesos do objetivo (só a proporção importa) ----
ALPHA, BETA, GAMMA = 0.5, 0.5, 0.0

# ---- adversários simulados ----
SEED             = 17
UNIFORM_OPPONENT = False  # False: sorteio "profissional" (tier + sinergia, padrão do model1.py) | True: uniforme
OPP_SHARP        = 2.0    # tau   (só vale se UNIFORM_OPPONENT=False)
OPP_SYNERGY      = 1.0    # sigma (só vale se UNIFORM_OPPONENT=False)

TOP_K = 5                 # nº de recomendações (draft) / times alternativos (game)

cfg = Config(n_players=N_PLAYERS, G=G, C=float(C), alpha=ALPHA, beta=BETA, gamma=GAMMA,
             allow_ubers=ALLOW_UBERS, E=E, opp_sharp=OPP_SHARP, opp_synergy=OPP_SYNERGY)

In [ ]:
# Dados embutidos (gzip + base64): costs_pokemon.csv e stats_pokemon.csv.
# Célula recolhida: não precisa abrir. Para usar seus próprios arquivos, defina USE_EMBEDDED_DATA = False.
_EMBEDDED = {
    "costs_pokemon.csv": """
H4sIAAAAAAACA5Wd2XbcOJKG7/ucfgffzQ3Vp+3equdOlix5kWy1Mkuq0h1IIpkoggwWCCjFfKN5jnmx+YO5yFVl68e4ysdaPoJY
ArEAAeTf/lq8+Xfx5ofizb+KN/8s3vyjePP34s3fijdvijevizd/LV7/u3j9Q/H6X8Xrfxav/1G8/nvx+m/F6zfF69fF678W/y5+
KP5V/LP4R/H3Ag8Vr//8p9NQ2TSeXAZJfV3sv7swLkyHb27EjdIXStT4d//TRbTWH745N6Etzp3xjTkW4SfXN8fvXLOOv/k+2GP5
41StXXX49kN1/M2tVG1xtnYnP6fDj94mlOGlw3/eFGeur20w+oRt3OjNuC7OvF2Z0tvirTdb19peeW/6V59db6PxdixO+96tnTcD
nutE+iaNo75gbfzcxHMUKVvxsXhrvXelfnVmvS0d3hOkNlWc0PZSOjP2ssFXwfz5T9cmjGtT4/t30YbexDQean1v8JPi05SC7U7u
1y7a4tIFgx459hfe2eDV17Yxr65SZYITbbrtT26MFBc+RRTx6tr0x965DAbV1q4Pxu0eRJPHiOGyxbmVpwmjNFhbF2cSHl3b6xgU
FzYEiWsJffHuqTJ1cN6jz6Qe5yHZddV10uGMUYq3Qfotekh7s7IDemgnAqZ3c03MrwlfQnC21q+KS+NRcS0AXRbQ0yh4GrXtZzYk
b+tGX2FaszVdcVpV1jcS0H2jePSftuDabuJGXv1cvJeTLxhM46dgn1QoTm4dxrq4Sj1ecmzufoR3PfLO2yoGiNL82wW4zvbVoTMO
/a8taYtL29sRePHZNKY3NYR5fuqT6Rsztmszi0FjhoTB/4BeePUWEwRidWzjwsumVZm+XIuvbd9IcendWKFFb/HvaKfiXW86CSrU
PUYUIgGpTNvtRvxxhM7tyvYYsDN012i9M8Vbg3GB1CX0zDZ1Zh6h925M+tIHCaKT7SxIaSCVsjJNcI2KWqpdL9oVcSe7p02Dav/5
T8VPkEZrxuImuM74V58maTDUxa6rD/U4jVG75WYNAe9kRA+bxmqdiyvT13Mjlvgd6oAhaFDNJZp+cpWKJbro1YXrHXoGU0i6YS/E
MqS+n4pLAd43r+Ypfxmsia+WaWz1S8jQL2aeWaNqE4Ofua4bexM85HSL97rBRbebj48Ok/u90WmAocPcML128VsTMA9au6lN8cli
HEQVUV85INK3+EkKGBA8bMKA+bxBo4pTN0iHnvn50XqohOLBtmh1UdyaCfK83YvXNdQFuhCT7NaOa8zXrlgIVJzHOx6mxoTa7iWt
cluM+q4rPkkrO4G5wyiaPoKRzYjxN6ji/IsleqEed19fYzIW79ErAT0L+ekgMGhJEBRTO8jJhZfQYCod5e7e2q2K3Xs3DLJRfXUG
WfAOX7w1Y4Vp1ujXV9K6J/RF09gwllNxNlXebdH4t2gI9GOzhq4onjXGqa8wO5xFt+ifr2fjT/qDz7YKsu0MlP2mP7lHDUb89Cgc
zxJ+K1Gg5VQXz4VgpIK+uEY58w+OcnIQ6oXpEpRSLD6ik2EL9jP1uclQch3UTDT9oQTvMWWL91MdrFO9eS6ox1b030Hn7jUm3rhR
HfVuHExco0/fdQPmnXbX2kLGkmpDlSJM/fSs+tBh3sa474Ub41vIOv48mM5sbR/NoW8OUjYP+hVkGD/2nVoYnVnFJxNQXYwkOgr9
7SK6KjqzH/X3eO24F56N6QYbDkpGklc1hz5StVT5YmcY1GTM+G5Wq/qCAD4rmMPcPJNuhVnxVYcd2pUaqEbZt+vBVErv/nyBhAwo
cylPMIlPxWJtps71J4t2Km4kTHj9G3QoZmMJ5VV8goFtHYYZY4N5fS0Hs/WVbbx2XiIU8X+SGohH9Mkl5Ay/1qofRvXBDLWMR5mo
7aM4tNxhhjlMind9tK44S9HZlZ922tg3x4F692SbJkUJ+yYd/txb86jz5scwwpxjBFRPl0GKGwsDH9I8IbW2Bzu0n717ga8dqO4o
7cUtzH9XWqsdYIb1dBTaS5E67DXF5aSzG215bzY+oYDiQkKwagJHrfigE/I8BdUG++pfWqlTbX9X+X2XFAtV2B5yAAOuJs0e6mc2
2raj6TnY2c/qz1hYoIV2HRyF+/m3v51+alRlMCc/9uXs7M2/Xk5hFpSAuj+pKBUXUHl+di1SnXrMHhhXzKJKh1ErcmgAJlP3u+oX
2utrt0onC7zfW3iJAQYakgXVUH8lpphXZuUgcmtUJP1Wf3x2tcyNm+1VUB9pqW7HSs353rZ8NZVgCTd7AbyzfRoN7OYHTGqL/1XW
HtHpCbblvTqKszaHRlA1cGhHMLACNvyhKfc69yHg9wYtuHNVVBu3aDGzMCeO4gV3AoKgXuRc8V+ThT9yC5NtSoFJWarKqEI6uBcL
dU0nOysDlX3XuXZKxUf1MyuwxV6sPvQr9SGH3/kbKsfHervuG7V+QDPVgjxA1KEhl/h2gAIZjyNzC2msDwOz09VqfjANxLvB6jPS
2BYOzL7G+9deTXD6glQn52q9r1M91sYfDKAaV1vDATpMhHloPqYmwRMqnif3wb80GDTT/qHy+ueoDxewR6Z7tTTOFz92ZdAxuzWz
L/FW/Ztn8wwt08zuKzSwQHmb4yQ9eqqfTAldMaAuavDCV3Ks1jmuv12VyT/qW5fwCYNSxY9P0Pi3GmvM4c8NHLYBlZ1/BCGCWvzS
Qgoad6zBFUQBGg0jewXFZHYeETpFBqkPVbiBhQb7zTrcmUHmpj9P1DvxmCuiqg3+DTQW/IPidj2hIg6K7Nnv/fJow69QGLMROiqr
Gwc3XOJsPM0w+KNGvTXQfembtSj2zs3b5JN6fq2JqNICNmHC4CNMUx/p0TwGlflPAifmRA5v6zE0kOcKw6C+6/PbYCAw37/7vqNq
OCiNYtHBeWtU/6GJ2+1B+k7hwwEongO8Wfq0vzXYO74ummi+9zZM97XrRlepG7KwbuxcFLPXkW+hTGYv/AYjFwUR1ZUqlXDUIdoR
4zrYzfeLR6FhP0DXdhwCHIK9bT15KK5c1TooABjcr0vUePalEqGQxqjWZjeNrxF6YgxbScUhgtyXdpf84J5eKgou0JBWK5QJhelh
blWia3U91PDFOdjbdfNLpfQSvIEDoZMDzuetTb2rPGzKtRmhGRGnoss6aOgXS0muSv2suoJp3SxnO6U5D8azHj+FW7Xz4787pntX
+WgY9q7NviS1giqwpx18RhT5QkkwAK15VLWzWKeqPSo+xMRRLfIpmu7i+sUSdNr2c3siXA/9uy9k312nOhlj6l8q5KcUWgeTjzkA
cuVUFTsEkGsj8/PuxaeL5WjwLDQHHDQ4hy3CxVLaFx+5l5N5HQIqv1fdFtz8kHeP5sXnfjIxFV/KEUpadmsGa/ty5YovHXSzaghl
oawJHeCWRT+vh0RXmqeXcbgctcVcWEIOZNzFyRpT0qcgamZeGnNq2QgOCbEInDEv8QCi7VpefuBWRshzbWc6IFS2hMfM2Mf7iFih
QseRPAEzalZRV/OEdP8CjoGayNMQpE21ITRGygxR129C1BkrnEd4WZyi+6HVCbyBQ6Qra1BGqWlehpfQyOPOxzp9elEHz3BoBfHI
6TapSqfwzm/9av1GozHylFrqGFIH2EeZXqbvoBO8T09aNOxXJMOpCx4ler7yOaAgnKaYrhkQvTFj8Fhr4dzkrV0xCoYcQsMpRH8I
/ii3s88UW1uYkY5ivk4ZkCoGyeFg7SVFCoYGPgcdVCi6YGlLMZGFjoHrc6QIHm6ZGk7FLS9LoA9DBgXTVlMK1j/U7hsh0O/BfuQS
JHB+4FT2kYP9uIHdYRx8nSfbc6orM+YeQlRnAm0EBEO4RMIv7Hs6CPO6P+3bkKJFNIbyeIekJ2pti7eIWTeUcVs+A5Ivjbp8lOtt
Rs8mtNCXHAsdZ3QFewUvjIBnpkLVOKRxCC9r3kahVGdDGiLFhhF+eU2xUCJQ5FQrnkO9e3QZzQy6UzBG3m0BofiaY2Ml1OKBiisJ
HcUw7oOjbbCItsZWAjXcZzbqAi+npK6patZVwgp+aMzg5tXEDO4Xww2Hcp2uwYcc0Or6PQej8GYguHRdBpVKx6lxcLxadhMzhgHh
OwJJw7lOV7MzMFuthWM9qJSBNbBWdExRWuV6+lY/L/R5jiXUjguINxs4ITncVHOVo1kEjhoGxVZ0sLyUpQws5inOZBds0rrNGxIZ
UBpb6oKciS5gml8MVTcShjW34MBG8YZTiL1/TS7wdoydUJ99pqhcSozS854LppQOznHpc9BgNp4LHWJ8R71QULr7S53QM10JpmsR
im1qTB5enJgmcVMdpIKAhgxMerOhWBoz9GGYBPE2nxKphAITTgl3ItIqYwRSlpRPfW0g5Kz254jSxzL1FAsxuCdOpc60rG7ntnJ1
QjhMudrycOHcQpFzy3BuuVk4t74yMU4ccxoAcmxYC+0zu4GU8dpvJEYODY67M+dr6zs30n79TTbC9ym+8gtK6ucEgBegxBnv3SOF
YuR68VxSqZkcDJtzIGCPOdd6MS3HusFwSHd8KWVrruxA2YFKdHCr0lOnVDHPoxJgvky8w9x26zzvf9lsLadSXbuGVy2VG6Ee2Pm8
zcJl/zd5Ld+n9okMlJPRZTQg7DJFKTdWnu5jKNb2woUbWPK828bI9d3Glty7eac72ponkMfBdaXgIxWid26sEoVa09OX+do2GXXS
nFGjiU05oHvkHus+C7XN46TO4Phqyzvf8JXtd10pNGZ812mOI4PGIQVakmZoVYmuaX8vk+uP3OAlsZ64MDCF7UipsLKxWv8XLy64
xgXDVtMvrOa0UMiWhlZN/a2WmvMLRG2aWRIDBzE3raUtRUiRmjWlerelUSWoni/VXHhjVquJU+X//g/+51wH68Mp3WbnVMipv4qQ
y1i92pEZHvKFdzFmVE9M5IvgF3M2VEZhmloZec9N3LZfSKfpIlTGd6cuGBXMU6L69kInKN3+VwxuK12/uEAw3Dr+TmlMxuaT5lJ2
EulLk60QNVMqrAJdpQMV6DLupSn5XuMx8zBvg/+Zz1sD+jq7OydiPfLZuvv4xBXCTh75P59e+X5K4Lfo8JdX167LLv1G+ilmN3ZO
1zRjdm00XXOQNrs2hzSiXP5n05l8+sE12zn/hz+QEQZeakZDTf0ETTWna23fzr/+A+W4m3zpOtidtSRPQbgR6k4wTo9a8S7zmo7N
oc5St+RS/YjAoa7jiiVjB/pSGpgvCvkyB6otdUaUQjDJKdtlMDGjUinwtwndKr6UYMuJRjCXCHLcylsa7F/CEPGtpR1lPZf7+Rhh
ToEpzBlxjNPZ4YzjWF/yOPhyzm2WKuOt307//z2m2dt0uRGYrpfyl34zd//3kECPT5zi/iKgxymH2szHRyiXypLGJ6C6ges6zUym
RaVuXAvd77hMoeZR6SVCfXi8bCTfwyHRLHaKDaj/RKngJhhOikXEfhlUDFRVvdeNHypjevKjE055p2cW2hxuzoemYE7m1nzAcZBI
I+bjaY+cbNEjnZlVdcBzN0wOfO7yzvE8WebsO/D/2Ti/4vHM8YwlbHxDd8iOtJ4/o/HlgV5Ow9rnLJweHrgTHyWUmfSDhEQnhYMc
99Xa9FmgtzaLizIwTt/a8qUfcHr8nVJCNybAwPDSFJf3As3Eax9Gy1+4OyeRhVEvZMbGyMOx91Bghu6dvp8GGph+aBo/lWnF1vA+
eJ8yNuU+wK7VrmN9+6FvzUSZmJPTtDs1rpc2MFX44XHKyHr8aPomw7h9dNpxA++4j3rsgbZCKbqZ8BFeg89449SzBZ/dkT8K1aYM
hlJt6jl0uDmCgiGYUs8hMtBWGcLxySHkn30CT0k9lBsoFbyjjfWGrmWC2awoA7+aOiCfPM8knZmcWslq5TKoztAVrE8YRpoU/Cm4
qrU8JXHPJaoXPwVpJQjtj1Su2FrhlXH0VBCgbuAhzZUZAnXUrkx4tDw9VzE96cowa1Z8elxpAiSVnitbO8OLqqeSVl7PcmIYG8rZ
gQeUV3Cm7cCh3RnjHG5IvLic1dIrh7hTOBTp+gygjaPrM1eiZ8ApNF86Q6loaMdLqgPdM7tKtavE026AW5GxH6Z3G2U4RlfzRRJ0
j+UqPdZurCj15Hj1nwJ1ZY73A1y7uv5/0fPFVIS/NqWMY3TUMbg2en/HkEHxRfKZ4kU11AaAqUyge5PAXGtCxhs7qhwVkhBzuN5m
HDybuUhFeMa2XIav4VFlLEZe610GGW3odxd/cY4vqOlx9ylDNIIevRk5Zk3vMkrjWv7aZJxxfOlmjd9Q4zpKxjQZq3XOnEuj3oFG
sQ1fGjrcwJOBxQypne+HOFzUlwePNDVtB2ZcGnBk57vZcsi8M8C7i33yjmvtLijK2Sw6Xq2T16jsWQe4d4mPqOenmear1GLGKzM2
cPX+GOqlKTQIlWxn+xVdXLtWOzdZGkheO19Zui6pW86vPoa/UCzrMA4wR8MwQDRJ/dqNte6VcNXoxg72jnMyX4VEGyCIf/m2ELCG
L56DGmzLXxnXNF03MztAsdsMLNUld8VAtXTp7zpRK6HnrjxNrAHGRTqFjDXS66S3NNLCPpvKuwxm1BstKRdZkPx5DugoVOvJnZML
2xlq4w7wdSbqeJfsMKFYX5maF9b/wnM8PksVZeMp5fg2PqBHTHFKjYiY6Xb459TRhbDPaestTen8PHU8++NLFeGVcXX9pa75Zs0X
17eIJhvqPn/pTD9RN+FLT08LfYEcNikkiiHM5Wn2X0Jcb/hR6C+Y5hlHaW70LtDgxkS53c0+9yIDXd0EPMxXe1Guz3BDQTUShFMD
3xgBNRq6Q6SUNbwsvhg3M7biozCOruOLYzd6hS0vbNO5DEZymIyX9Y4vsd3YEEYbWi46L1zT9zUFhUC3rW7WkIkpA4qJRmk3a8cD
7puX7vY7MvPNgHkU3VqZuYlD8sjr3vQ81rtxrclpI1wj21LK764PplxfpaA54By0Fe8xvSuRQoOnS7g3Po3UdN2I0xuRKeXdxjQ5
FLS1z+GgJ9ac05sNc/LGsvJpb0SqdUbsdSN6lJIO1O7mRkZplpcZLMf6nCENUkqG/3MzThnZjjdQK61eUUu5IWMn5iaF4IVPAjga
Dc3MvZkCP6V0M6U2dfhLuP8k80TfqJBeOEcx582jyaC8qWlnZObv3OogRQ6NlaFQhgm4NT6OlOl050o4l5W6zi6CfcZevsB1R9nG
0VOeCuE/6iEpF4TOpVvr0YCe6rRbvTMuUEivgKDn/W7XEz9/CkhzRhjlxFOxQC8EnpVxK01j+8DPftyKnyqhE/NWpHU1zdXSSzR5
ysJ8SW4Oc3JBnbw9F4Rap1vEyDRr/hZdyym99bivHcd2n6ZAsb42lfR0OUBBvm6fcwXykfNcK8x3Szc8R2ZhNi1HxpKbxEU1f7gI
v7PqcOkvxQSF8d3ohX62CUwGtRh6k2vT8OKmyG9LWlhT07QnQC1PlQHFEwEW1tbUjAHyFOkjP962sCHjuriFfXRDRk9tTMYFaocr
8Clma11e45jnDqZSnh8+nTFqrZXi53sWa2d9nVEzt4p0TQyU64Vv54Cj2X6LdRDJGSa9op67XYt1GgbeG3Aj/DTQwpzPuEdwoTfA
C09GXLjOZSwnKZaxnjRjfEFpd9e85XXLPJq5cNvt7uMmGAgHgM8qxPQZB8+A8YuFFq0EBD2UCtYPFNIUZpp7tmg38shlzWcpY1D8
vHHeYYDMo6QLnxp6tGXRCb8zCBCidLpTnHc4YdG7RzrWPZeZXgK/Qg3UI9fDfSozTq4tJONuloXogYuRUxnxy2LIubsC1IaeTVgM
a0sva1sMrjfBtBkYd04HFxx88ZJywlOIF0PQCzUcXdpUMPILkRa/JninpoR3N3HUBX6QbwF1knGt1nw3/yONiBTjNQM0JQpZy29W
XETpfxG9r5g3QFLUaIdyiJzowWs9497SdqZaNnAh6NinfqU3ZnCs5XuaixRGvrm82JiShuiA6ox7wYB5yXgh39NfbBwUGoeCp7uQ
C3Q83yZemhrqs6SU04+OoFRf287Q7BP96CdLlzOW+tEd88eMUTCOqYHKolziqWVLW9cOAkRfOn8GGb9xbWn5ed0l/P9HOuGW8LLX
lNmYiqdjLl1X8kO9u4+k4hp+x/GFrCWUsSR6Ddbuk0cyelYaq+JG99kVpL6nQjxoWuoV4jlUMBWfehKifhwTxaJkrBUtJVVQMFRX
LeXJ8q59clXgdz6C01xFR/3/ZTC93tnqOTfotdkUs7aiqWSgHm3PT3Asgww8ZW45H9AfedVSV1recwlzQQ8mhAxwwzXI1GXsOi4n
/WRTCg32v1995i519gHl5RSkSbxyGSHBj/1KP8KYYtDfgcd1u08J4mO65+huyp0JPF6+sz5tDYX6nDgemGTchKVYT9XRnd64/2ui
J4vuXBn4jp1+tidmaUl1zZ3TZB/eBmjAwSeazHkHP2DLZfLO6fIwx8RjMkdOZZy8v0v6+aUTpwbq2t3DHeN3nSjF7+S4Nz5Yukx+
b6CvMmKpexOrNb00/j7rZO29tXztERAsRsk3mu81kSJnUgEcK76XolhHFwzv59PeGc7Rfdal1Peu44nz+jHFXH3fQ++5DdrJPKN7
KfWYP11Lu88JNwB5ukaWlXN4L6EzNU0mvM+6yvg+hY5fYHkP+0k/c+x+CrWltc+68e3n+bM8KNTThUJlLHW8f059wy8EejAzxfrq
wZb6kcD0AsLcy+weRBNYWk7RW00eEhfTh42F7kzj/wFiU6ksG4YAAA==
""",
    "stats_pokemon.csv": """
...
LpeOnf8fN+Dy21RRAQA=
""",
}


def embedded_csv(name):
    raw = gzip.decompress(base64.b64decode("".join(_EMBEDDED[name].split())))
    return io.StringIO(raw.decode("utf-8"))

## 2. Dados
Os dados vêm dos CSVs embutidos (tabela de custos e estatísticas extraídas da Smogon). `load_data` junta a tabela de custos (formato largo: uma coluna por custo) com as estatísticas, converte nomes (ex.: `Mega Tyranitar` → `Tyranitar-Mega`), calcula o poder de ataque $a_i=\max(\text{Atk}_i,\text{SpA}_i)$ e o normaliza em $v_i\in[0,1]$.

In [ ]:
if USE_EMBEDDED_DATA:
    costs_src, stats_src = embedded_csv("costs_pokemon.csv"), embedded_csv("stats_pokemon.csv")
else:
    costs_src, stats_src = COSTS_CSV, STATS_CSV

df, missing, n_illegal = load_data(costs_src, stats_src, ATTACK_MODE, BAN_ILLEGAL)
pool = len(df) - (0 if ALLOW_UBERS else int(df["banned"].sum()))

print(f"{len(df)} Pokémon carregados | {int(df['banned'].sum())} Ubers/AG "
      f"({'liberados' if ALLOW_UBERS else 'banidos'}) | pool efetivo: {pool}")
if BAN_ILLEGAL:
    print(f"{n_illegal} Pokémon do tier Illegal removidos do pool.")
if missing:
    print(f"Aviso: {len(missing)} nomes do arquivo de custos sem stats foram ignorados.")
assert N_PLAYERS * G <= pool, f"Pool insuficiente: {N_PLAYERS}x{G} > {pool}"

df.head(10)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

ax[0, 0].hist(df["cost"], bins=range(int(df["cost"].min()), int(df["cost"].max()) + 2),
              color=COLORS[0], edgecolor="white")
ax[0, 0].set(title="Distribuição de custos", xlabel="custo (pontos)", ylabel="nº de Pokémon")

tiers = df["Tier"].value_counts()
ax[0, 1].bar(tiers.index, tiers.values, color=COLORS[1])
ax[0, 1].set(title="Pokémon por tier", ylabel="nº de Pokémon")
ax[0, 1].tick_params(axis="x", rotation=60)

type_counts = pd.Series([t for ts in df["types"] for t in ts]).value_counts()
ax[1, 0].barh(type_counts.index[::-1], type_counts.values[::-1], color=COLORS[2])
ax[1, 0].set(title="Frequência de cada tipo no pool", xlabel="nº de Pokémon")

ax[1, 1].scatter(df["cost"], df["attack"], s=12, alpha=0.6,
                 c=df["mega"].map({True: "tab:red", False: "tab:blue"}))
ax[1, 1].set(title="Poder de ataque × custo (vermelho = Mega)", xlabel="custo", ylabel="max(Atk, SpA)")

plt.tight_layout()
plt.show()

print(f"Correlação custo × ataque: {df[['cost', 'attack']].corr().iloc[0, 1]:.2f}  |  Megas no pool: {int(df['mega'].sum())}")

## 3. Tipos: `Advantage` e `Resist`
Para um tipo atacante $t$ e um Pokémon defensor $j$, o multiplicador de dano é $m(t,j)=\prod_{q\in T_j} f(t,q)$ com $f\in\{0,\tfrac12,1,2\}$. O escore do tipo é

$$
s(t,j)=\begin{cases}+2 & m(t,j)>1\\ -2 & m(t,j)<1\ (\text{resistência ou imunidade})\\ 0 & m(t,j)=1\end{cases}
$$

e a vantagem de $i$ sobre $j$ soma os tipos de $i$ (ataque com bônus de mesmo tipo, STAB):

$$\text{Advantage}(i,j)=\sum_{t\in T_i}s(t,j),\qquad \text{Resist}(i,j)=-\text{Advantage}(j,i).$$

Abaixo, a tabela de eficácia usada pelo código (ataque nas linhas, defesa nas colunas).

In [ ]:
mult = np.array([[type_multiplier(a, (d,)) for d in TYPES] for a in TYPES])

cmap = ListedColormap(["#444444", "#e57373", "#f5f5f5", "#81c784"])      # 0, 1/2, 1, 2
norm = BoundaryNorm([-0.25, 0.25, 0.75, 1.5, 2.5], cmap.N)

fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(mult, cmap=cmap, norm=norm)
ax.set_xticks(range(len(TYPES)), TYPES, rotation=90)
ax.set_yticks(range(len(TYPES)), TYPES)
ax.set(xlabel="tipo defensor", ylabel="tipo atacante", title="Multiplicadores de dano (tipos puros)")
for a in range(len(TYPES)):
    for d in range(len(TYPES)):
        if mult[a, d] != 1:
            ax.text(d, a, f"{mult[a, d]:g}", ha="center", va="center", fontsize=7,
                    color="white" if mult[a, d] == 0 else "black")
plt.tight_layout()
plt.show()

**Exemplo passo a passo** (o mesmo dos slides): Mega Tyranitar (Rocha/Sombrio) contra Typhlosion de Hisui (Fogo/Fantasma). Note a assimetria: `Resist` olha o confronto no sentido contrário.

In [ ]:
def explain(atk_types, def_types):
    print(f"{'/'.join(atk_types)}  atacando  {'/'.join(def_types)}")
    for t in atk_types:
        m = type_multiplier(t, def_types)
        s = 2 if m > 1 else (-2 if m < 1 else 0)
        print(f"   {t:<9} -> x{m:g}   =>  s = {s:+d}")
    print(f"   Advantage = {advantage(atk_types, def_types):+d}\n")

tyranitar, typhlosion = ("Rock", "Dark"), ("Fire", "Ghost")
explain(tyranitar, typhlosion)                      # ataque de i sobre j
explain(typhlosion, tyranitar)                      # ataque de j sobre i
print(f"Resist(Tyranitar, Typhlosion) = -Advantage(Typhlosion, Tyranitar) = "
      f"{resistance(tyranitar, typhlosion):+d}")
print(f"d_i para um único adversário: {advantage(tyranitar, typhlosion) / 4:+.2f}   "
      f"r_i: {resistance(tyranitar, typhlosion) / 4:+.2f}   (normalização: 4|U|, com |U| = 1)")

## 4. O modelo MILP

### Draft (rodada $M$)
Conjuntos: $N$ (candidatos), $A$ (meus Pokémon), $U$ (Pokémon dos adversários), Ubers (banidos), Mega. Parâmetros: $w_i$ (custo), $C$ (orçamento), $E$ (máx. de Megas), $G$ (tamanho do time).

$$
\begin{aligned}
\max\quad & Z=\sum_{i\in N}\left(\alpha v_i+\beta d_i+\gamma r_i\right)x_i\\
\text{s.a.}\quad & \sum_{i\in N}x_i=M\\
& \sum_{i\in N}w_i x_i\le B_M,\qquad B_M=C-\sum_{k=1}^{G-M}w_{(k)}\\
& \sum_{i\in \text{Mega}}x_i\le E\\
& x_i=1\quad\forall i\in A\\
& x_i=0\quad\forall i\in (U\cup\text{Ubers})\setminus A\\
& x_i\in\{0,1\}\quad\forall i\in N
\end{aligned}
$$

com $d_i=\dfrac{\sum_{j\in U}\text{Advantage}(i,j)}{4|U|}$ e $r_i=\dfrac{\sum_{j\in U}\text{Resist}(i,j)}{4|U|}$, ambos em $[-1,1]$, na mesma escala de $v_i\in[0,1]$. A reserva $B_M$ guarda o custo dos $G-M$ Pokémon mais baratos ainda livres, para garantir que o time possa ser completado.

### Game
É o **mesmo MILP** com $M=G$ e $B_M=C$: os $n-1$ times adversários já existem e resolvemos uma única vez.

### Duelo
Depois do draft, escolhe-se $K$ Pokémon do roster $R$ contra um adversário $O$, sem custos:
$\max\ \sum_{i\in R}(\beta d_i+\gamma r_i)x_i$ s.a. $\sum x_i=K$, $\sum_{i\in\text{Mega}}x_i\le 1$, $x_i\in\{0,1\}$ (com $d_i,r_i$ calculados sobre $O$).

Os problemas são resolvidos por `scipy.optimize.milp` (solver HiGHS, branch-and-cut).

## 5. Coeficientes $c_i$ contra um conjunto de adversários
Sorteamos $n-1$ times adversários (disjuntos entre si, dentro do orçamento e do limite de Megas) e calculamos, para cada candidato, $v_i$, $d_i$, $r_i$ e $c_i$. Depois de calculados, os $c_i$ são **constantes**: por isso o objetivo é linear.

In [ ]:
def team_table(df, idx, c=None, adv_t=None, res_t=None):
    """Tabela descritiva de um conjunto de Pokémon (índices do df)."""
    idx = list(idx)
    out = pd.DataFrame({
        "Pokémon": df["name"].values[idx],
        "Tipos": ["/".join(df["types"].values[i]) for i in idx],
        "Tier": df["Tier"].values[idx],
        "Custo": df["cost"].values[idx].astype(int),
        "Atk": df["attack"].values[idx].astype(int),
        "v": df["v"].values[idx].round(2),
    })
    if adv_t is not None:
        out["d"] = adv_t[idx].round(2)
        out["r"] = res_t[idx].round(2)
    if c is not None:
        out["c"] = c[idx].round(3)
    return out.reset_index(drop=True)


def sample_opponents(df, cfg, rng, uniform):
    opp, taken = [], set()
    for _ in range(cfg.n_players - 1):
        team = random_team(df, taken, cfg, rng, uniform)
        opp.append(team)
        taken |= set(team)
    return opp


def allowed_pool(df, U, allow_ubers):
    """Índices que ainda podem ser escolhidos (não estão em U e não são banidos)."""
    blocked = set(U)
    ok = [i for i in range(len(df))
          if i not in blocked and (allow_ubers or not df["banned"].iloc[i])]
    return np.array(ok)

In [ ]:
rng = np.random.default_rng(SEED)
opp = sample_opponents(df, cfg, rng, UNIFORM_OPPONENT)
U = [j for team in opp for j in team]

for k, team in enumerate(opp, 1):
    print(f"Adversário {k}  (custo {df['cost'].values[team].sum():.0f}/{C},  "
          f"Megas: {int(df['mega'].values[team].sum())})")
    display(team_table(df, team))

In [ ]:
c, adv_t, res_t = objective_coeffs(df, U, cfg)
allowed = allowed_pool(df, U, ALLOW_UBERS)

top = allowed[np.argsort(-c[allowed])][:15]
print("15 maiores coeficientes c_i (ignorando custo e limite de Megas):")
display(team_table(df, top, c, adv_t, res_t))

fig, ax = plt.subplots(figsize=(8, 5.5))
sc = ax.scatter(df["v"].values[allowed], adv_t[allowed], c=res_t[allowed], cmap="coolwarm",
                vmin=-1, vmax=1, s=16, alpha=0.8)
ax.set(xlabel="força de ataque  $v_i$", ylabel="vantagem ofensiva  $d_i$",
       title="Os três critérios para cada candidato (cor = resistência $r_i$)")
plt.colorbar(sc, label="$r_i$")
plt.tight_layout()
plt.show()

## 6. Modo Game: o melhor time contra adversários prontos
Resolvemos o MILP uma vez ($M=G$, $A=\emptyset$). Os **Top-$k$ times** vêm de cortes *no-good*: para cada time já encontrado $T$, adiciona-se $\sum_{i\in T}x_i\le |T|-1$, o que proíbe repeti-lo.

In [ ]:
game_teams = best_teams(df, U, cfg, top_k=TOP_K)
assert game_teams, "Modelo inviável: orçamento C insuficiente para G Pokémon."

best, Z_best = game_teams[0]
best = sorted(best, key=lambda i: -c[i])

print(f"MELHOR TIME  (Z = {Z_best:.3f}, custo {df['cost'].values[best].sum():.0f}/{C})")
display(team_table(df, best, c, adv_t, res_t))

print("\nTimes alternativos:")
alt = pd.DataFrame([{"#": k, "Z": round(z, 3), "Custo": int(df["cost"].values[t].sum()),
                     "Time": ", ".join(df["name"].values[sorted(t, key=lambda i: -c[i])])}
                    for k, (t, z) in enumerate(game_teams, 1)])
display(alt)

**Verificação das restrições** e comparação com a solução que *ignora o orçamento* (os $G$ maiores $c_i$, respeitando só o limite de Megas). A diferença mostra o papel da restrição de custo, que torna o problema uma mochila com cardinalidade.

In [ ]:
def greedy_select(mega_flags, scores, k, E):
    """Os k maiores escores respeitando no máx. E Megas. Devolve posições."""
    chosen, megas = [], 0
    for p in np.argsort(-np.asarray(scores), kind="stable"):
        if mega_flags[p]:
            if E is not None and megas >= E:
                continue
            megas += 1
        chosen.append(int(p))
        if len(chosen) == k:
            break
    return chosen


# --- verificação ---
cost_best = df["cost"].values[best].sum()
megas_best = int(df["mega"].values[best].sum())
assert len(best) == G
assert cost_best <= C
assert megas_best <= E
assert not set(best) & set(U)
assert ALLOW_UBERS or not df["banned"].values[best].any()
print(f"OK: |T|={len(best)}, custo={cost_best:.0f} <= {C}, Megas={megas_best} <= {E}, disjunto de U, sem Ubers.")

# --- comparação com a solução sem orçamento ---
pos = greedy_select(df["mega"].values[allowed], c[allowed], G, E)
free = [int(allowed[p]) for p in pos]
print(f"\n{'':<24}{'Z':>8}{'custo':>8}")
print(f"{'MILP (com orçamento)':<24}{Z_best:>8.3f}{cost_best:>8.0f}")
print(f"{'top-G por c_i (sem C)':<24}{c[free].sum():>8.3f}{df['cost'].values[free].sum():>8.0f}"
      f"   {'<-- viola o orçamento' if df['cost'].values[free].sum() > C else '(cabe no orçamento)'}")

In [ ]:
types = df["types"].values
off = np.array([[sum(advantage(types[i], types[j]) for j in t) for t in opp] for i in best])
dfn = np.array([[sum(resistance(types[i], types[j]) for j in t) for t in opp] for i in best])
lim = 4 * G

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, mat, title in zip(axes, (off, dfn), ("Ofensa: Σ Advantage(i, j)", "Defesa: Σ Resist(i, j)")):
    im = ax.imshow(mat, cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(len(opp)), [f"Adv {k}" for k in range(1, len(opp) + 1)])
    ax.set_yticks(range(len(best)), df["name"].values[best])
    ax.set_title(title)
    for a in range(mat.shape[0]):
        for b in range(mat.shape[1]):
            ax.text(b, a, f"{mat[a, b]:+d}", ha="center", va="center", fontsize=9)
plt.suptitle("Confronto de tipos do meu time contra cada time adversário", y=1.02)
plt.tight_layout()
plt.show()

## 7. Modo Draft: simulação de um draft completo
A cada rodada $M$: cada adversário escolhe um Pokémon; o MILP é resolvido com $M$ integrantes (mantendo $A$ fixo, com reserva de orçamento) e devolve as $k$ melhores recomendações. Aqui o "jogador" sempre aceita a **recomendação #1**, o que permite ver o comportamento do modelo sem interação.

In [ ]:
def simulate_draft(df, cfg, rng, uniform=True, top_k=5):
    n_opp = cfg.n_players - 1
    A, opp_d, log = [], [[] for _ in range(n_opp)], []
    for rnd in range(1, cfg.G + 1):
        picks = []
        for k in range(n_opp):
            taken = set(A) | {j for t in opp_d for j in t}
            spent = df["cost"].values[opp_d[k]].sum() if opp_d[k] else 0.0
            j = opponent_pick(df, taken, cfg, rng, uniform, budget_left=cfg.C - spent,
                              slots_left=cfg.G - len(opp_d[k]), team=opp_d[k])
            opp_d[k].append(j)
            picks.append(j)

        U_d = [j for t in opp_d for j in t]
        recs = recommend(df, A, U_d, rnd, cfg, top_k=top_k)
        if not recs:
            print(f"Rodada {rnd}: modelo inviável.")
            break
        pick, Z, _ = recs[0]
        A.append(pick)
        log.append({
            "Rodada": rnd,
            "Adversários escolheram": ", ".join(df["name"].values[picks]),
            "Recomendação #1": df["name"].iloc[pick],
            "Tipos": "/".join(df["types"].iloc[pick]),
            "Custo": int(df["cost"].iloc[pick]),
            "Gasto acumulado": int(df["cost"].values[A].sum()),
            "Z": round(Z, 3),
            "Alternativas": ", ".join(df["name"].iloc[i] for i, _, _ in recs[1:]),
        })
    return A, opp_d, log


A_draft, opp_draft, draft_log = simulate_draft(df, cfg, np.random.default_rng(SEED),
                                               UNIFORM_OPPONENT, TOP_K)
draft_df = pd.DataFrame(draft_log)
display(draft_df)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(draft_df["Rodada"], draft_df["Gasto acumulado"], marker="o", color=COLORS[0], label="gasto acumulado")
ax.axhline(C, color="tab:red", ls="--", label=f"orçamento C = {C}")
ax.set(xlabel="rodada", ylabel="pontos gastos", title="Orçamento ao longo do draft")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Meu time final: custo {df['cost'].values[A_draft].sum():.0f}/{C}, "
      f"Megas {int(df['mega'].values[A_draft].sum())}/{E}")

**Como ficou meu time contra cada adversário?** Para cada time adversário final, a vantagem ofensiva média ($d$) e a resistência média ($r$) do meu time, ambas em $[-1,1]$, calculadas sobre todos os pares (meu Pokémon, Pokémon dele).

In [ ]:
def type_balance(df, X, Y):
    types = df["types"].values
    d = np.mean([advantage(types[i], types[j]) for i in X for j in Y]) / 4
    r = np.mean([resistance(types[i], types[j]) for i in X for j in Y]) / 4
    return d, r


rows = []
for k, team in enumerate(opp_draft, 1):
    d, r = type_balance(df, A_draft, team)
    rows.append({"Adversário": k, "Custo dele": int(df["cost"].values[team].sum()),
                 "d (meu ataque)": round(d, 2), "r (minha defesa)": round(r, 2)})
display(pd.DataFrame(rows))

print("Meu time:")
display(team_table(df, A_draft))

## 8. Sensibilidade aos pesos $(\alpha,\beta,\gamma)$
Para cada semente sorteamos **um** conjunto de adversários e resolvemos o modo game em quatro cenários: $\alpha$ dominante, $\beta$ dominante, $\gamma$ dominante e pesos iguais. Como o conjunto de adversários é o mesmo, qualquer diferença entre os times vem só dos pesos.

> $Z$ **não é comparável** entre cenários (os pesos mudam a escala do objetivo). Comparamos pelas métricas do time, que não dependem dos pesos.

In [ ]:
DOMINANT = 0.70     # peso do parâmetro dominante nos 3 primeiros cenários
N_SEEDS  = 10       # nº de conjuntos de adversários sorteados
SEED0    = 0        # primeira semente

scen = make_scenarios(DOMINANT)
names = list(scen)
print(pd.DataFrame(scen, index=["alpha", "beta", "gama"]).T.round(2))

In [ ]:
sens_teams, rows = {}, []
for s in range(SEED0, SEED0 + N_SEEDS):
    opp_s = sample_opponents(df, cfg, np.random.default_rng(s), UNIFORM_OPPONENT)
    U_s = [j for t in opp_s for j in t]
    _, adv_s, res_s = objective_coeffs(df, U_s, cfg)       # termos de tipo não dependem dos pesos

    for name, (a, b, g) in scen.items():
        cfg_s = replace(cfg, alpha=a, beta=b, gamma=g)
        chosen, Z = solve(df, [], U_s, cfg_s.G, cfg_s)      # modo game: M = G, A = vazio
        if chosen is None:
            continue
        sens_teams[(s, name)] = chosen
        m = team_metrics(df, chosen, U_s, adv_s, res_s)
        rows.append({"semente": s, "cenario": name, "Z": Z, **m,
                     "time": ";".join(df["name"].values[chosen])})

res_df = pd.DataFrame(rows)
print(f"{len(res_df)} resoluções ({res_df['semente'].nunique()} sementes x {len(names)} cenários)")

In [ ]:
summary = res_df.groupby("cenario")[METRIC_COLS].mean().reindex(names).round(2)
summary.columns = ["custo", "atk", "d", "r", "%of+", "%df-", "tipos", "megas"]
display(summary)
print("d, r: vantagem ofensiva / resistência médias (escala [-1,1])."
      "\n%of+: % dos pares (meu, adversário) com vantagem ofensiva | %df-: % dos pares em que sou atingido super efetivamente.")

In [ ]:
panels = [("custo", "Custo do time"), ("atk_medio", "Ataque médio"), ("d_medio", "$d$ médio"),
          ("r_medio", "$r$ médio"), ("pares_ofensivos_%", "% pares com vantagem ofensiva"),
          ("pares_defensivos_ruins_%", "% pares defensivos ruins")]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (col, title) in zip(axes.flat, panels):
    g_ = res_df.groupby("cenario")[col]
    mean, std = g_.mean().reindex(names), g_.std().reindex(names).fillna(0)
    ax.bar(range(len(names)), mean.values, yerr=std.values, capsize=3, color=COLORS)
    ax.set_xticks(range(len(names)), [n.replace(" dominante", "\ndom.") for n in names])
    ax.set_title(title)
plt.suptitle("Métricas por cenário (média ± desvio entre sementes)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5))
for name, col in zip(names, COLORS):
    sub = res_df[res_df["cenario"] == name]
    ax.scatter(sub["custo"], sub["d_medio"], color=col, s=45, alpha=0.8, label=name)
ax.set(xlabel="custo do time", ylabel="$d$ médio (vantagem ofensiva)",
       title="Trade-off custo × vantagem ofensiva (cada ponto = uma semente)")
ax.legend()
plt.tight_layout()
plt.show()

**Quanto os times de cada cenário se parecem?** Número médio de Pokémon em comum (de $G$) entre os times de dois cenários, sobre as mesmas sementes. E, ao lado, a presença de cada tipo nos times recomendados.

In [ ]:
overlap = pd.DataFrame(index=names, columns=names, dtype=float)
for a, b in itertools.product(names, names):
    vals = [len(set(sens_teams[(s, a)]) & set(sens_teams[(s, b)]))
            for s in range(SEED0, SEED0 + N_SEEDS)
            if (s, a) in sens_teams and (s, b) in sens_teams]
    overlap.loc[a, b] = np.mean(vals) if vals else np.nan

share = pd.DataFrame(0.0, index=names, columns=TYPES)
for name in names:
    ts = [t for (s, n), t in sens_teams.items() if n == name]
    cnt = Counter(tp for t in ts for i in t for tp in df["types"].values[i])
    for tp in TYPES:
        share.loc[name, tp] = cnt[tp] / max(len(ts) * G, 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1, 2]})
im = axes[0].imshow(overlap.values.astype(float), cmap="Blues", vmin=0, vmax=G)
axes[0].set_xticks(range(len(names)), [n.replace(" dominante", "\ndom.") for n in names])
axes[0].set_yticks(range(len(names)), names)
axes[0].set_title(f"Pokémon em comum (de {G})")
for a in range(len(names)):
    for b in range(len(names)):
        axes[0].text(b, a, f"{overlap.values[a, b]:.1f}", ha="center", va="center",
                     color="white" if overlap.values[a, b] > G / 2 else "black")

im2 = axes[1].imshow(share.values, cmap="Oranges", aspect="auto")
axes[1].set_xticks(range(len(TYPES)), TYPES, rotation=60)
axes[1].set_yticks(range(len(names)), names)
axes[1].set_title("Tipos nos times recomendados (ocorrências por vaga)")
plt.colorbar(im2, ax=axes[1], fraction=0.025)
plt.tight_layout()
plt.show()

## 9. Duelo: escolher $K$ do roster contra um adversário (`model2.py`)
Depois do draft, cada batalha pede um problema menor: sem custos, só o roster $R$, um adversário $O$ e exatamente $K$ escolhas, com no máximo uma Mega. Para o roster usamos a união dos Top-$k$ times do modo game (seção 6) e, como adversário, o primeiro time sorteado.

Como cada escore $s_i=\beta d_i+\gamma r_i$ não depende dos demais Pokémon do time, o problema é **separável**: o ótimo é escolher os $K$ maiores $s_i$ (respeitando o limite de Megas). Verificamos isso comparando o MILP com a seleção gulosa.

In [ ]:
roster = list(dict.fromkeys(i for t, _ in game_teams for i in t))   # união dos Top-k times
O = opp[0]
K = min(4, len(roster))
print(f"Roster: {len(roster)} Pokémon | adversário: {len(O)} Pokémon | K = {K}\n")

_, d_all, r_all = objective_coeffs(df, O, cfg)      # d_i e r_i calculados sobre O
roster_tbl = team_table(df, roster, None, d_all, r_all)[["Pokémon", "Tipos", "Custo", "d", "r"]]
display(roster_tbl)

In [ ]:
rows = []
for beta, gamma in [(1.0, 0.0), (0.5, 0.5), (0.0, 1.0)]:
    chosen_pos, Z, score = best_team(df, roster, O, K, beta, gamma, E)
    team = [roster[k] for k in chosen_pos]

    g_pos = greedy_select(df["mega"].values[roster], score, K, E)       # seleção gulosa
    Z_greedy = float(np.asarray(score)[g_pos].sum())

    tc = Counter(t for i in team for t in df["types"].values[i])
    top_t, top_n = tc.most_common(1)[0]
    rows.append({"β": beta, "γ": gamma, "Z (MILP)": round(Z, 3), "Z (guloso)": round(Z_greedy, 3),
                 "iguais?": abs(Z - Z_greedy) < 1e-9,
                 "Time": ", ".join(df["name"].values[team]),
                 "tipos distintos": len(tc), "tipo mais repetido": f"{top_t} ×{top_n}"})

duel_df = pd.DataFrame(rows)
display(duel_df)

Para usar **nomes digitados** em vez de Pokémon sorteados, converta-os para índices do `df`. A função abaixo usa o nome exato quando existe e avisa quando precisa aproximar.

In [ ]:
def names_to_idx(df, names):
    idx = []
    for n in names:
        i = find_pokemon(df, n)
        if i is None:
            raise ValueError(f"'{n}' não encontrado no arquivo de custos.")
        if df["name"].iloc[i].lower() != n.strip().lower():
            print(f"Aviso: '{n}' interpretado como '{df['name'].iloc[i]}'.")
        if i not in idx:
            idx.append(i)
    return idx


# Exemplo (troque pelos nomes do seu roster e do adversário, como aparecem em df["name"]):
# R_manual = names_to_idx(df, ["Terrakion", "Diancie", "Kleavor", "Archeops", "Mega Lucario"])
# O_manual = names_to_idx(df, ["Corviknight", "Gholdengo"])
# chosen_pos, Z, score = best_team(df, R_manual, O_manual, 4, beta=1.0, gamma=0.5, max_mega=E)
# print(df["name"].values[[R_manual[k] for k in chosen_pos]], round(Z, 3))

## 10. Discussão

**Limitações**
- **Separabilidade.** Como $c_i$ depende só do Pokémon $i$ e não dos pares dentro do time, o modelo não vê redundância: pode recomendar tipos repetidos (a tabela do duelo e o mapa de tipos da seção 8 mostram isso). A restrição de orçamento e o limite de Megas são os únicos termos que acoplam as escolhas.
- **Aproximações de combate.** Ataques são representados só pelos tipos (STAB); golpes de cobertura, velocidade, habilidades e itens ficam de fora.
- **Rodada 1.** Sem adversários conhecidos ($U=\emptyset$), $d_i=r_i=0$ e o modelo se reduz ao critério puro de poder, ponderado por $\alpha$.
- **Pesos fixos.** $(\alpha,\beta,\gamma)$ expressam a estratégia e não são otimizados; só a proporção importa.

**Generalização natural: sinergia entre Pokémon do mesmo time.** Penalizar a sobreposição de tipos leva a uma Programação Quadrática Binária (MIQP):

$$
\max\ \sum_{i\in N} c_i x_i\;-\;\lambda\sum_{i<k}\text{Overlap}(T_i,T_k)\,x_i x_k
$$

Isso diversifica o time, ao custo de perder a linearidade (e, no duelo, a equivalência com "ordenar por escore"). Uma alternativa que mantém o problema linear é linearizar os produtos $x_i x_k$ com variáveis auxiliares $y_{ik}\ge x_i+x_k-1$.